In [1]:
"""
============================================================================
 QUANTUM GENE REGULATORY NETWORK INFERENCE — run across ALL THREE datasets
 Drosophila melanogaster ovary: germline vs. somatic (follicle) gene panel
============================================================================

Same pipeline as the single-dataset script, run once per dataset:
    FlyCellAtlas, GSE136162, GSE146040

Genes (6 total -> 6 qubits):
    Germline (qubits 0-2): Gurken (grk), BicaudalD (BicD), egalitarian (egl)
    Somatic  (qubits 3-5): Torpedo (top), kekkon1 (kek1), spitz (spi)

For each dataset this script:
    1. Loads real per-cell counts and binarizes expression (count > 0).
    2. Computes per-gene activation ratios -> encoder (Ry) angles.
    3. Builds the observed joint distribution P_obs over 2^6 = 64 states
       (intra-population structure real; cross-population combined by
       independence, since there's no shared pairing key -- see the
       original single-dataset script's docstring for the full rationale).
    4. Builds the 30-edge fully-connected candidate circuit and trains
       the CRY angles with exact analytic gradient descent, minimizing
       D_KL(P_model || P_obs).
    5. Rebuilds the trained circuit in real Qiskit, runs it on the free
       local AerSimulator.
    6. Saves four outputs *per dataset*, each filename tagged with the
       dataset name so the three runs never overwrite each other:
           quantum_circuit_diagram_<dataset>.png
           weighted_grn_network_<dataset>.png
           training_loss_curve_<dataset>.png
           learned_edge_angles_<dataset>.csv

No "known biology" edges are hard-coded anywhere -- all 30 candidate
edges are treated identically at initialization and during training.
============================================================================
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator


# ============================================================================
# CONFIG
# ============================================================================
BASE_FOLDER = r"C:\Users\13306\Desktop\CS-495"

DATASET_FILES = {
    "FlyCellAtlas": os.path.join(BASE_FOLDER, "FlyCellAtlas_ovary_gene_panel_v2.xlsx"),
    "GSE136162":    os.path.join(BASE_FOLDER, "GSE136162_ovary_gene_panel.xlsx"),
    "GSE146040":    os.path.join(BASE_FOLDER, "GSE146040_ovary_gene_panel.xlsx"),
}

GERMLINE_GENES = ["Gurken_grk", "BicaudalD_BicD", "egalitarian_egl"]
SOMATIC_GENES  = ["Torpedo_top", "kekkon1_kek1", "spitz_spi"]
ALL_GENES = GERMLINE_GENES + SOMATIC_GENES          # qubit index == position in this list
SHORT_NAME = {
    "Gurken_grk": "grk", "BicaudalD_BicD": "BicD", "egalitarian_egl": "egl",
    "Torpedo_top": "top", "kekkon1_kek1": "kek1", "spitz_spi": "spi",
}
NGENES = len(ALL_GENES)                              # 6 qubits -> 64 basis states
NSTATES = 2 ** NGENES

DROP_ZERO = True          # exclude the |000000> state and rescale (qscGRN default)
EPOCHS = 2000
LEARNING_RATE = 0.15
LOSS_REL_THRESHOLD = 1e-10   # stop early once relative loss change is this small
LOSS_WINDOW = 10
LAPLACE_ALPHA = 1.0

SHOTS = 8192               # AerSimulator shots for the final executed circuit
EDGE_PERCENTILE_FOR_BINARY = 75   # percentile of |phi| used ONLY for the binary/reported edge list

# ============================================================================
# GROUND-TRUTH BIOLOGY (for the validation column only -- never used in
# training, only to score the trained edges after the fact)
# ============================================================================
# Each tuple is (control_gene_short, target_gene_short). Source: Grk/EGFR
# and transport-complex literature (SDB Online gene briefs; Clark 2007 for
# the Egl/BicD motor-complex transport of grk).
EXPECTED_EDGES = {
    ("grk", "top"):  "Grk is the main ligand for EGFR (germline -> somatic)",
    ("spi", "top"):  "Spitz is an EGFR ligand, autocrine within follicle cells",
    ("top", "kek1"): "kek1 is a regulatory target of EGFR signaling",
    ("kek1", "top"): "kek1 is a negative-feedback inhibitor of EGFR",
    ("egl", "grk"):  "Egl is part of the motor complex transporting grk into the oocyte",
    ("BicD", "grk"): "BicD is part of the motor complex transporting grk into the oocyte",
    ("egl", "BicD"): "Egl and BicD co-localize in the oocyte (transport complex partners)",
    ("BicD", "egl"): "Egl and BicD co-localize in the oocyte (transport complex partners)",
}


# ============================================================================
# SECTION 1 -- LOAD DATA & BINARIZE
# ============================================================================
def load_and_binarize(path):
    """
    Loads one ovary gene panel and returns, for each of the two populations,
    a binary activation matrix (cells x genes) where an entry is 1 if the
    raw UMI count for that gene in that cell is > 0. Threshold-at-zero
    convention matches qscGRN's qsc_distribution().
    """
    df = pd.read_excel(path)

    germline_df = df.loc[df["Cell_Type"] == "germline", GERMLINE_GENES]
    somatic_df  = df.loc[df["Cell_Type"] == "somatic",  SOMATIC_GENES]

    germline_binary = (germline_df.to_numpy() > 0).astype(int)   # (n_germline, 3)
    somatic_binary  = (somatic_df.to_numpy()  > 0).astype(int)   # (n_somatic, 3)

    return germline_binary, somatic_binary


# ============================================================================
# SECTION 2 -- ACTIVATION RATIOS -> Ry ENCODER ANGLES
# ============================================================================
def activation_ratios(binary_matrix):
    """p_j = fraction of cells in which gene j is active."""
    return binary_matrix.mean(axis=0)


def ry_angles_from_ratios(p):
    """theta_j = 2 * arcsin(sqrt(p_j))  -- so that P(qubit_j = 1) = p_j."""
    return 2.0 * np.arcsin(np.sqrt(np.clip(p, 0.0, 1.0)))


# ============================================================================
# SECTION 3 -- OBSERVED JOINT DISTRIBUTION P_obs OVER 64 STATES
# ============================================================================
def population_joint_distribution(binary_matrix):
    """
    Exact empirical joint distribution over the 2^3 = 8 activation states
    of a 3-gene population, computed directly from real per-cell joint
    measurements. Bit i of the state index corresponds to column i of
    binary_matrix (qubit-0-as-least-significant-bit, matching Qiskit).
    """
    n_cells, n_local_genes = binary_matrix.shape
    n_local_states = 2 ** n_local_genes
    counts = np.zeros(n_local_states)
    weights = 1 << np.arange(n_local_genes)          # [1, 2, 4]
    state_ids = binary_matrix @ weights               # (n_cells,)
    for s in state_ids:
        counts[s] += 1
    return counts / n_cells


def build_joint_observed_distribution(germline_binary, somatic_binary):
    """
    Combines the two REAL intra-population joint distributions into one
    64-state distribution over all 6 genes, by independence (no shared
    pairing key exists between the two populations in this dataset).
    Qubits 0-2 = germline genes, qubits 3-5 = somatic genes.
    """
    p_germ = population_joint_distribution(germline_binary)   # length 8
    p_som  = population_joint_distribution(somatic_binary)    # length 8

    p_obs = np.zeros(NSTATES)
    for gs in range(8):
        for ss in range(8):
            idx = gs | (ss << 3)
            p_obs[idx] = p_germ[gs] * p_som[ss]

    if DROP_ZERO:
        p_obs[0] = 0.0
        p_obs = p_obs / p_obs.sum()

    return p_obs


# ============================================================================
# SECTION 4 -- QUANTUM GATE MATH (Ry, CRY, and their derivatives)
# ============================================================================
def ry_gate(theta):
    c, s = np.cos(theta / 2.0), np.sin(theta / 2.0)
    return np.array([[c, -s], [s, c]])


def der_ry_gate(theta):
    c, s = np.cos(theta / 2.0), np.sin(theta / 2.0)
    return 0.5 * np.array([[-s, -c], [c, -s]])


def _controlled_gate_matrix(nqubits, control, target, target_block):
    """Builds a full 2^n x 2^n matrix applying `target_block` to `target`
    only when `control` is |1>, identity elsewhere."""
    zero_proj = np.array([[1.0, 0.0], [0.0, 0.0]])
    one_proj  = np.array([[0.0, 0.0], [0.0, 1.0]])
    RR, II = None, None
    for i in range(nqubits):
        if i == control:
            fac_RR, fac_II = one_proj, zero_proj
        elif i == target:
            fac_RR, fac_II = target_block, np.identity(2)
        else:
            fac_RR, fac_II = np.identity(2), np.identity(2)
        RR = np.kron(fac_RR, RR) if RR is not None else fac_RR
        II = np.kron(fac_II, II) if II is not None else fac_II
    return RR, II


def cry_gate(theta, nqubits, control, target):
    RR, II = _controlled_gate_matrix(nqubits, control, target, ry_gate(theta))
    return RR + II


def der_cry_gate(theta, nqubits, control, target):
    RR, _ = _controlled_gate_matrix(nqubits, control, target, der_ry_gate(theta))
    return RR


def tensor_product(matrices):
    """matrices given in qubit-index order [q0, q1, ..., q_{n-1}]."""
    arr = None
    for m in matrices:
        arr = np.kron(m, arr) if arr is not None else m
    return arr


def matrix_multiplication(matrices):
    """Given [T1, T2, ..., Tk] (applied in that order to the state),
    returns Tk @ ... @ T2 @ T1."""
    arr = None
    for m in matrices:
        arr = m @ arr if arr is not None else m
    return arr


# ============================================================================
# SECTION 5 -- CANDIDATE EDGES: FULLY-CONNECTED, 30 DIRECTED CRY GATES
# ============================================================================
def build_edges(genes):
    """All ordered pairs (g_i, g_j), i != j  ->  n^2 - n edges (30 for n=6)."""
    return [(a, b) for a in genes for b in genes if a != b]


# ============================================================================
# SECTION 6 -- TRAINING: exact analytic gradient descent
# ============================================================================
def laplace_smooth(distribution, ncells, alpha=LAPLACE_ALPHA):
    dim = len(distribution)
    scaled = ncells * distribution
    n_total = scaled.sum()
    scaled = scaled + alpha
    n_total = n_total + alpha * dim
    return scaled / n_total, n_total


def loss_constraint(phi_values, ratio):
    """Boundary-avoidance regularizer, keeping edge angles away from the
    +/- pi/2 singularity."""
    return np.sum(ratio / np.power(phi_values ** 4 - (np.pi / 2) ** 4, 2))


def train(p_obs, theta_fixed, edges, indexes, ncells_for_smoothing,
          epochs=EPOCHS, lr=LEARNING_RATE):
    """
    theta_fixed : length-6 array, the Ry encoder angles (NOT trained).
    edges       : list of 30 (gene_a, gene_b) tuples.
    indexes     : list of 30 (control_qubit, target_qubit) tuples.
    Returns the trained phi array (length 30) and the loss history.
    """
    n_edges = len(edges)
    phi = np.zeros(n_edges)   # start with NO assumed interactions

    encoder_mats = [ry_gate(theta_fixed[i]) for i in range(NGENES)]
    encoder = tensor_product(encoder_mats)          # fixed throughout training
    input_state = np.zeros((NSTATES, 1))
    input_state[0, 0] = 1.0

    p_obs_col = p_obs.reshape(NSTATES, 1)
    loss_history = []

    for epoch in range(epochs):
        # ---- forward pass: exact statevector ----
        regulation_mats = [cry_gate(phi[i], NGENES, indexes[i][0], indexes[i][1])
                            for i in range(n_edges)]
        transform_reg = matrix_multiplication(regulation_mats)
        v = transform_reg @ encoder @ input_state             # (64,1) amplitude vector
        p_out = v ** 2

        if DROP_ZERO:
            p_out = p_out.copy()
            p_out[0, 0] = 0.0
            p_out = p_out / p_out.sum()

        # ---- Laplace-smoothed probabilities for a numerically stable KL ----
        h_p_out, h_N_out = laplace_smooth(p_out.flatten(), ncells_for_smoothing)
        h_p_obs, h_N_obs = laplace_smooth(p_obs_col.flatten(), ncells_for_smoothing)
        h_p_out = h_p_out.reshape(NSTATES, 1)
        h_p_obs = h_p_obs.reshape(NSTATES, 1)

        loss = float(np.sum(h_p_out * np.log(h_p_out / h_p_obs)))     # D_KL(model || obs)
        loss_ratio = loss / loss_constraint(phi, ratio=1.0)
        cons = loss_constraint(phi, ratio=loss_ratio)
        loss_history.append(loss)

        # ---- analytic gradient ----
        log_term = np.log(h_p_out / h_p_obs)
        der_const = (-2 * loss_ratio * 4 * phi ** 3) / np.power(phi ** 4 - (np.pi / 2) ** 4, 3)
        gradient = np.zeros(n_edges)

        for i in range(n_edges):
            control, target = indexes[i]
            der_reg_mats = regulation_mats.copy()
            der_reg_mats[i] = der_cry_gate(phi[i], NGENES, control, target)
            der_transform_reg = matrix_multiplication(der_reg_mats)
            der_state = der_transform_reg @ encoder @ input_state       # (64,1)

            der_loss = (1 + log_term) * 2 * v * der_state * (NSTATES / h_N_out)
            gradient[i] = np.sum(der_loss) + der_const[i]

        # ---- early stopping on relative loss change ----
        if epoch >= LOSS_WINDOW + 1:
            recent = np.mean(loss_history[epoch - (LOSS_WINDOW - 1): epoch + 1])
            past = loss_history[epoch - LOSS_WINDOW]
            delta = abs(past - recent) / abs(recent) if recent != 0 else 0.0
            if delta < LOSS_REL_THRESHOLD:
                print(f"[train] Converged at epoch {epoch} (delta={delta:.2e})")
                break

        # ---- gradient descent step ----
        phi = phi - lr * gradient

        if epoch % 100 == 0:
            print(f"[train] epoch {epoch:4d}  KL loss = {loss:.6f}")

    print(f"[train] final KL loss = {loss_history[-1]:.6f} after {len(loss_history)} epochs")
    return phi, loss_history


# ============================================================================
# SECTION 7 -- REBUILD IN REAL QISKIT & RUN ON A FREE SIMULATOR (AerSimulator)
# ============================================================================
def build_qiskit_circuit(theta_fixed, phi_trained, edges, indexes):
    """Builds the actual, literal Qiskit circuit: 6 Ry gates (encoder) +
    30 trained CRY gates (regulation) + measurement, all 6 qubits."""
    circ = QuantumCircuit(NGENES, NGENES)

    for q in range(NGENES):
        circ.ry(theta_fixed[q], q)
    circ.barrier()

    for i, (control, target) in enumerate(indexes):
        circ.cry(phi_trained[i], control, target)
    circ.barrier()

    circ.measure(range(NGENES), range(NGENES))
    return circ


def run_on_simulator(circ, shots=SHOTS):
    """Executes the circuit on Qiskit's free, local AerSimulator."""
    backend = AerSimulator()
    compiled = transpile(circ, backend)
    job = backend.run(compiled, shots=shots)
    counts = job.result().get_counts()
    return counts


def counts_to_probabilities(counts, shots):
    """Qiskit bitstrings are ordered qN-1...q1q0 (qubit 0 = rightmost
    character), matching our state-index convention directly."""
    p_model = np.zeros(NSTATES)
    for bitstring, n in counts.items():
        idx = int(bitstring, 2)
        p_model[idx] = n / shots
    return p_model


# ============================================================================
# SECTION 8a -- OUTPUT: FULL CIRCUIT DIAGRAM
# ============================================================================
def save_circuit_diagram(circ, path):
    fig = circ.draw(output="mpl", fold=-1, style={"fontsize": 10})
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"[output] full circuit diagram -> {path}")


# ============================================================================
# SECTION 8b -- OUTPUT: WEIGHTED GRN NETWORK DIAGRAM
# ============================================================================
def save_weighted_network(edges, phi_trained, path, dataset_name):
    G = nx.DiGraph()
    for gene in ALL_GENES:
        G.add_node(SHORT_NAME[gene])

    max_abs_phi = np.max(np.abs(phi_trained)) if np.max(np.abs(phi_trained)) > 0 else 1.0

    pos = {}
    germ_short = [SHORT_NAME[g] for g in GERMLINE_GENES]
    som_short  = [SHORT_NAME[g] for g in SOMATIC_GENES]
    for i, g in enumerate(germ_short):
        pos[g] = (-1.0, 1.0 - i)
    for i, g in enumerate(som_short):
        pos[g] = (1.0, 1.0 - i)

    fig, ax = plt.subplots(figsize=(9, 7))

    node_colors = ["#e7a352" if g in germ_short else "#5fa8d0" for g in G.nodes()]
    nx.draw_networkx_nodes(G, pos, node_size=1800, node_color=node_colors,
                            edgecolors="#20242c", linewidths=1.5, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=12, font_weight="bold", ax=ax)

    for (gene_a, gene_b), phi_val in zip(edges, phi_trained):
        a, b = SHORT_NAME[gene_a], SHORT_NAME[gene_b]
        magnitude = abs(phi_val)
        alpha = float(np.clip((magnitude / max_abs_phi) ** 1.3, 0.02, 1.0))
        width = float(np.clip(0.4 + 5.0 * (magnitude / max_abs_phi), 0.4, 5.4))
        same_population = (gene_a in GERMLINE_GENES) == (gene_b in GERMLINE_GENES)
        rad = 0.15 if same_population else 0.08
        color = "#c96a3a" if phi_val >= 0 else "#3a7fc9"

        nx.draw_networkx_edges(
            G, pos, edgelist=[(a, b)], ax=ax,
            width=width, alpha=alpha, edge_color=color,
            connectionstyle=f"arc3,rad={rad}",
            arrowstyle="-|>", arrowsize=14, node_size=1800,
        )

    ax.set_title(f"Inferred gene regulatory network — {dataset_name}\n"
                 f"(edge weight = |trained CRY angle|)", fontsize=12)
    ax.text(-1.0, 1.8, "germline", ha="center", fontsize=10, color="#8a5a20")
    ax.text(1.0, 1.8, "somatic", ha="center", fontsize=10, color="#2a5a80")
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"[output] weighted network diagram -> {path}")


# ============================================================================
# SECTION 8c -- OUTPUT: TRAINING CURVE + EDGE TABLE
# ============================================================================
def save_training_curve(loss_history, path, dataset_name):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(loss_history, color="#5fa8d0")
    ax.set_xlabel("epoch")
    ax.set_ylabel("D_KL(P_model || P_obs)")
    ax.set_title(f"Training loss — {dataset_name}")
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"[output] training curve -> {path}")


def save_edge_table(edges, phi_trained, path, dataset_name):
    threshold = np.percentile(np.abs(phi_trained), EDGE_PERCENTILE_FOR_BINARY)
    rows = []
    for (gene_a, gene_b), phi_val in zip(edges, phi_trained):
        a_short, b_short = SHORT_NAME[gene_a], SHORT_NAME[gene_b]
        above_threshold = abs(phi_val) >= threshold
        is_expected = (a_short, b_short) in EXPECTED_EDGES

        # Validation logic:
        #   expected edge  + model says "connected"    -> Correct (found a real connection)
        #   expected edge  + model says "not connected" -> Incorrect (missed a real connection)
        #   unexpected edge + model says "not connected" -> Correct (correctly found no relation)
        #   unexpected edge + model says "connected"    -> Incorrect (flagged a connection that isn't known to exist)
        if is_expected:
            validation = "Correct" if above_threshold else "Incorrect"
        else:
            validation = "Correct" if not above_threshold else "Incorrect"

        rows.append({
            "control_gene": a_short,
            "target_gene": b_short,
            "edge": f"{a_short} -> {b_short}",
            "population_pair": (
                "intra-germline" if gene_a in GERMLINE_GENES and gene_b in GERMLINE_GENES else
                "intra-somatic" if gene_a in SOMATIC_GENES and gene_b in SOMATIC_GENES else
                "cross-population"
            ),
            "trained_phi_radians": phi_val,
            "abs_phi": abs(phi_val),
            f"above_{EDGE_PERCENTILE_FOR_BINARY}th_percentile": above_threshold,
            "known_biological_relationship": EXPECTED_EDGES.get((a_short, b_short), "(none known)"),
            "validation": validation,
        })
    out = pd.DataFrame(rows).sort_values("abs_phi", ascending=False)

    score = int((out["validation"] == "Correct").sum())
    n_total = len(out)

    out.to_csv(path, index=False)
    print(f"[output] all 30 learned edge angles -> {path}")
    print(out.to_string(index=False))
    print(f"\n[validation] {dataset_name}: {score}/{n_total} edges match known biology "
          f"(score = {score}/{n_total})")
    return out, score


# ============================================================================
# RUN ONE DATASET, START TO FINISH
# ============================================================================
def run_pipeline_for_dataset(dataset_name, excel_path):
    print("\n" + "#" * 70)
    print(f"# DATASET: {dataset_name}")
    print(f"# FILE:    {excel_path}")
    print("#" * 70)

    out_circuit  = os.path.join(BASE_FOLDER, f"quantum_circuit_diagram_{dataset_name}.png")
    out_network  = os.path.join(BASE_FOLDER, f"weighted_grn_network_{dataset_name}.png")
    out_curve    = os.path.join(BASE_FOLDER, f"training_loss_curve_{dataset_name}.png")
    out_edges    = os.path.join(BASE_FOLDER, f"learned_edge_angles_{dataset_name}.csv")

    print("\n" + "=" * 70)
    print("STEP 1: loading and binarizing real single-cell data")
    print("=" * 70)
    germline_binary, somatic_binary = load_and_binarize(excel_path)
    print(f"germline cells: {germline_binary.shape[0]}   somatic cells: {somatic_binary.shape[0]}")

    print("\n" + "=" * 70)
    print("STEP 2: activation ratios -> Ry encoder angles (fixed, from data)")
    print("=" * 70)
    p_germ = activation_ratios(germline_binary)
    p_som = activation_ratios(somatic_binary)
    p_all = np.concatenate([p_germ, p_som])           # order matches ALL_GENES
    theta_fixed = ry_angles_from_ratios(p_all)
    for gene, p, th in zip(ALL_GENES, p_all, theta_fixed):
        print(f"  {SHORT_NAME[gene]:5s}  p={p:.4f}   theta={th:.4f} rad")

    print("\n" + "=" * 70)
    print("STEP 3: building the observed joint distribution P_obs (64 states)")
    print("=" * 70)
    p_obs = build_joint_observed_distribution(germline_binary, somatic_binary)
    print(f"P_obs sums to {p_obs.sum():.6f}, nonzero states: {np.count_nonzero(p_obs)}/64")

    print("\n" + "=" * 70)
    print("STEP 4: building the 30 fully-connected candidate CRY edges")
    print("=" * 70)
    edges = build_edges(ALL_GENES)
    indexes = [(ALL_GENES.index(a), ALL_GENES.index(b)) for a, b in edges]
    print(f"{len(edges)} candidate directed edges (expected 30)")

    print("\n" + "=" * 70)
    print("STEP 5: training (exact analytic gradient descent, qscGRN-style)")
    print("=" * 70)
    ncells_for_smoothing = min(germline_binary.shape[0], somatic_binary.shape[0])
    phi_trained, loss_history = train(p_obs, theta_fixed, edges, indexes,
                                       ncells_for_smoothing)

    print("\n" + "=" * 70)
    print("STEP 6: rebuilding the trained circuit in Qiskit and running it")
    print("         on the free, local AerSimulator")
    print("=" * 70)
    circ = build_qiskit_circuit(theta_fixed, phi_trained, edges, indexes)
    counts = run_on_simulator(circ, shots=SHOTS)
    p_model_sampled = counts_to_probabilities(counts, SHOTS)
    print(f"Executed circuit with {SHOTS} shots on AerSimulator; "
          f"{len(counts)} distinct states observed.")

    print("\n" + "=" * 70)
    print("STEP 7: saving outputs")
    print("=" * 70)
    save_circuit_diagram(circ, out_circuit)
    save_weighted_network(edges, phi_trained, out_network, dataset_name)
    save_training_curve(loss_history, out_curve, dataset_name)
    edge_table, validation_score = save_edge_table(edges, phi_trained, out_edges, dataset_name)

    print(f"\nDone with {dataset_name}. Outputs written:")
    print(f"  - {out_circuit}")
    print(f"  - {out_network}")
    print(f"  - {out_curve}")
    print(f"  - {out_edges}")

    return {
        "germline_cells": germline_binary.shape[0],
        "somatic_cells": somatic_binary.shape[0],
        "final_kl_loss": loss_history[-1],
        "phi_trained": phi_trained,
        "edge_table": edge_table,
        "validation_score": validation_score,
    }


# ============================================================================
# MAIN — loop over all three datasets
# ============================================================================
def main():
    results = {}
    for dataset_name, excel_path in DATASET_FILES.items():
        results[dataset_name] = run_pipeline_for_dataset(dataset_name, excel_path)

    print("\n\n" + "#" * 70)
    print("# ALL THREE DATASETS COMPLETE — summary")
    print("#" * 70)
    summary_rows = []
    for name, r in results.items():
        summary_rows.append({
            "Dataset": name,
            "Germline cells": r["germline_cells"],
            "Somatic cells": r["somatic_cells"],
            "Final KL loss": round(r["final_kl_loss"], 6),
            "Validation score (out of 30)": r["validation_score"],
        })
    print(pd.DataFrame(summary_rows).to_string(index=False))


if __name__ == "__main__":
    main()


######################################################################
# DATASET: FlyCellAtlas
# FILE:    C:\Users\13306\Desktop\CS-495\FlyCellAtlas_ovary_gene_panel_v2.xlsx
######################################################################

STEP 1: loading and binarizing real single-cell data
germline cells: 3267   somatic cells: 3425

STEP 2: activation ratios -> Ry encoder angles (fixed, from data)
  grk    p=0.1246   theta=0.7215 rad
  BicD   p=0.5507   theta=1.6723 rad
  egl    p=0.2914   theta=1.1404 rad
  top    p=0.6038   theta=1.7799 rad
  kek1   p=0.3442   theta=1.2540 rad
  spi    p=0.2193   theta=0.9746 rad

STEP 3: building the observed joint distribution P_obs (64 states)
P_obs sums to 1.000000, nonzero states: 63/64

STEP 4: building the 30 fully-connected candidate CRY edges
30 candidate directed edges (expected 30)

STEP 5: training (exact analytic gradient descent, qscGRN-style)
[train] epoch    0  KL loss = 0.143951
[train] epoch  100  KL loss = 0.114391
[train]

In [1]:
"""
============================================================================
 QUANTUM GRN -- SYNTHETIC SPATIAL TEST  (positive-control experiment)
 Input: FlyCellAtlas_ovary_gene_panel_v2.xlsx  
============================================================================

PURPOSE
    No public dataset says which germline cell touches which follicle cell, so
    the normal pipeline must assume the two populations are independent
    (P_obs = P_germline x P_follicle). That makes every cross-population edge,
    including Grk -> EGFR, unlearnable by construction. This script asks:
    IF that pairing existed, would the pipeline pick it up?

WHAT IS REAL AND WHAT IS SYNTHETIC
    Real:       every cell's expression (no value is ever changed), every gene's
                activation rate, and the joint distribution of the 3 genes inside
                the germline cells and inside the follicle cells.
    Synthetic:  the egg-chamber geometry (x, y positions) and WHICH follicle cell
                touches which germline cell (one-to-one contact pairs).

    Two scenarios use the SAME cells and the SAME geometry. They differ only in
    who touches whom:
        null     contacts are random.             Cross-population edges should stay small.
        planted  contacts are biased so that germline cells with the first gene of a
                 PLANTED_LINK ON tend to touch follicle cells with the second gene ON
                 (default: grk -> top). The pipeline SHOULD now find that edge.

HOW TO READ THE RESULT (important)
    This is a positive control. A hit in "planted" shows the pipeline CAN detect
    germline-follicle coupling when it exists at a known strength. It does NOT show
    that Grk -> EGFR coupling exists in real flies: the coupling was put there by
    this script. Do not present planted results as biological findings.
    Coupling between two genes is symmetric, so the data cannot say whether the
    edge points grk -> top or top -> grk; the script reports both.

WHAT CHANGED FROM THE ORIGINAL PIPELINE (everything else is identical)
    1. Step 1 builds the synthetic spatial data (instead of just loading cells).
    2. P_obs is counted from the contact pairs (6 genes observed together in each
       pair) instead of multiplying two independent distributions.
    3. Ry angles come from the same paired observations, so the circuit's single-
       gene probabilities match P_obs.
    4. Laplace smoothing uses the number of contact pairs as its cell count.
    Training, Qiskit execution, plots, edge table and validation are unchanged.

OUTPUTS (written to BASE_FOLDER, tagged with the scenario name)
    synthetic_spatial_pairs_<scenario>.csv      the synthetic contact pairs + coordinates
    synthetic_chamber_example.png               one chamber, null vs planted
    quantum_circuit_diagram_FlyCellAtlas_SYNTHETIC_<scenario>.png
    weighted_grn_network_FlyCellAtlas_SYNTHETIC_<scenario>.png
    training_loss_curve_FlyCellAtlas_SYNTHETIC_<scenario>.png
    learned_edge_angles_FlyCellAtlas_SYNTHETIC_<scenario>.csv
============================================================================
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator


# ============================================================================
# CONFIG
# ============================================================================
BASE_FOLDER = r"C:\Users\13306\Desktop\CS-495"
INPUT_XLSX = os.path.join(BASE_FOLDER, "FlyCellAtlas_ovary_gene_panel_v2.xlsx")

GERMLINE_GENES = ["Gurken_grk", "BicaudalD_BicD", "egalitarian_egl"]
SOMATIC_GENES  = ["Torpedo_top", "kekkon1_kek1", "spitz_spi"]
ALL_GENES = GERMLINE_GENES + SOMATIC_GENES          # qubit index == position in this list
SHORT_NAME = {
    "Gurken_grk": "grk", "BicaudalD_BicD": "BicD", "egalitarian_egl": "egl",
    "Torpedo_top": "top", "kekkon1_kek1": "kek1", "spitz_spi": "spi",
}
NGENES = len(ALL_GENES)                              # 6 qubits -> 64 basis states
NSTATES = 2 ** NGENES

DROP_ZERO = True          # exclude the |000000> state and rescale (qscGRN default)
EPOCHS = 2000
LEARNING_RATE = 0.15
LOSS_REL_THRESHOLD = 1e-10   # stop early once relative loss change is this small
LOSS_WINDOW = 10
LAPLACE_ALPHA = 1.0

SHOTS = 8192               # AerSimulator shots for the final executed circuit
EDGE_PERCENTILE_FOR_BINARY = 75   # percentile of |phi| used ONLY for the binary/reported edge list

# ---- Synthetic spatial data settings (NEW) --------------------------------
SEED = 495
PAIRS_PER_CHAMBER = 16     # each synthetic egg chamber: 16 germline cells, each touching 1 follicle cell
R_GERM = 1.0               # germline cells fill a disk of this radius at the chamber centre
R_FOLL = 1.6               # follicle cells form a ring of this radius around them
CHAMBER_SPACING = 5.0      # distance between chamber centres (only matters for the picture)

# (germline gene, somatic gene) pairs whose ON states are made to co-occur across contacts
PLANTED_LINKS = [("grk", "top")]
COUPLING_STRENGTH = 2.0    # 0 = none. Larger = stronger coupling (Metropolis inverse temperature)
N_SWAP_STEPS = 400_000     # swap proposals used to build the planted contacts

# scenario name -> coupling strength
SCENARIOS = {"null": 0.0, "planted": COUPLING_STRENGTH}

# ============================================================================
# GROUND-TRUTH BIOLOGY (for the validation column only -- never used in
# training, only to score the trained edges after the fact)
# ============================================================================
EXPECTED_EDGES = {
    ("grk", "top"):  "Grk is the main ligand for EGFR (germline -> somatic)",
    ("spi", "top"):  "Spitz is an EGFR ligand, autocrine within follicle cells",
    ("top", "kek1"): "kek1 is a regulatory target of EGFR signaling",
    ("kek1", "top"): "kek1 is a negative-feedback inhibitor of EGFR",
    ("egl", "grk"):  "Egl is part of the motor complex transporting grk into the oocyte",
    ("BicD", "grk"): "BicD is part of the motor complex transporting grk into the oocyte",
    ("egl", "BicD"): "Egl and BicD co-localize in the oocyte (transport complex partners)",
    ("BicD", "egl"): "Egl and BicD co-localize in the oocyte (transport complex partners)",
}


# ============================================================================
# SECTION 1 -- LOAD REAL CELLS, THEN BUILD SYNTHETIC SPATIAL DATA   (CHANGED)
# ============================================================================
def load_panel(path):
    """
    Real cells only. Returns binary activation (count > 0, same convention as
    before) for germline cells (3 genes) and somatic cells (3 genes), plus ids.
    """
    df = pd.read_excel(path)
    id_col = "Cell_Barcode" if "Cell_Barcode" in df.columns else "Cell"
    germ = df[df["Cell_Type"] == "germline"].reset_index(drop=True)
    soma = df[df["Cell_Type"] == "somatic"].reset_index(drop=True)
    return {
        "germ_bits": (germ[GERMLINE_GENES].to_numpy() > 0).astype(int),
        "soma_bits": (soma[SOMATIC_GENES].to_numpy() > 0).astype(int),
        "germ_ids": germ[id_col].astype(str).to_numpy(),
        "soma_ids": soma[id_col].astype(str).to_numpy(),
    }


def build_synthetic_world(real, rng):
    """
    Synthetic geometry. Chambers sit on a grid; in each one the germline cells
    fill a small disk and the follicle cells form a ring around it. Every
    follicle cell is matched to its nearest free germline cell (one-to-one,
    minimum total distance), giving K contact pairs per chamber.

    This step never looks at gene expression. It only fixes which real cells
    are used, where their slots are, and which slots touch.
    """
    k = PAIRS_PER_CHAMBER
    n_chambers = min(len(real["germ_ids"]), len(real["soma_ids"])) // k
    n_pairs = n_chambers * k
    germ_sel = rng.permutation(len(real["germ_ids"]))[:n_pairs]   # real germline cells used
    soma_sel = rng.permutation(len(real["soma_ids"]))[:n_pairs]   # real follicle cells used

    grid = int(np.ceil(np.sqrt(n_chambers)))
    germ_xy = np.zeros((n_pairs, 2))
    foll_xy = np.zeros((n_pairs, 2))
    dist = np.zeros(n_pairs)
    chamber = np.repeat(np.arange(n_chambers), k)

    for c in range(n_chambers):
        cx, cy = (c % grid) * CHAMBER_SPACING, (c // grid) * CHAMBER_SPACING
        r = R_GERM * np.sqrt(rng.random(k))
        th = rng.uniform(0, 2 * np.pi, k)
        g_xy = np.c_[cx + r * np.cos(th), cy + r * np.sin(th)]
        ang = np.arange(k) * 2 * np.pi / k + rng.uniform(0, 2 * np.pi / k)
        rad = R_FOLL + rng.normal(0, 0.08, k)
        f_xy = np.c_[cx + rad * np.cos(ang), cy + rad * np.sin(ang)]
        g_idx, f_idx = linear_sum_assignment(cdist(g_xy, f_xy))     # nearest one-to-one contacts
        sl = slice(c * k, (c + 1) * k)
        germ_xy[sl], foll_xy[sl] = g_xy[g_idx], f_xy[f_idx]
        dist[sl] = np.linalg.norm(g_xy[g_idx] - f_xy[f_idx], axis=1)

    return {"n_chambers": n_chambers, "n_pairs": n_pairs, "germ_sel": germ_sel,
            "soma_sel": soma_sel, "germ_xy": germ_xy, "foll_xy": foll_xy,
            "dist": dist, "chamber": chamber}


def assign_follicle_cells(g_link_bits, f_link_bits, strength, n_steps, rng):
    """
    Decide which real follicle cell sits in each contact slot.

    strength = 0 -> a random shuffle (null scenario).
    strength > 0 -> Metropolis swaps. A swap moves two follicle cells between two
    slots. Score M = number of contacts where, for every planted link, the
    germline gene AND the follicle gene are both ON. A swap is accepted with
    probability min(1, exp(strength * change_in_M)).

    Swaps only rearrange the SAME cells, so every gene's activation rate and the
    joint distribution within each population stay exactly as in the real data.
    Only the cross-population pairing changes.
    """
    n = g_link_bits.shape[0]
    occ = rng.permutation(n)                      # occ[slot] = index of follicle cell in that slot
    if strength <= 0 or n_steps <= 0:
        return occ
    g = g_link_bits.astype(np.int64)
    f = f_link_bits.astype(np.int64)
    a_idx = rng.integers(0, n, n_steps)
    b_idx = rng.integers(0, n, n_steps)
    u = rng.random(n_steps)
    for t in range(n_steps):
        a, b = a_idx[t], b_idx[t]
        if a == b:
            continue
        delta = int(np.dot(f[occ[b]] - f[occ[a]], g[a] - g[b]))    # change in M if swapped
        if delta >= 0 or u[t] < np.exp(strength * delta):
            occ[a], occ[b] = occ[b], occ[a]
    return occ


def make_paired_observations(real, world, strength, rng):
    """One scenario: returns the 6-gene observation for every contact pair."""
    germ_bits = real["germ_bits"][world["germ_sel"]]                 # (n_pairs, 3)
    pool_bits = real["soma_bits"][world["soma_sel"]]                 # (n_pairs, 3) follicle cells available
    g_short = [SHORT_NAME[g] for g in GERMLINE_GENES]
    s_short = [SHORT_NAME[g] for g in SOMATIC_GENES]
    gcols = [g_short.index(a) for a, _ in PLANTED_LINKS]
    scols = [s_short.index(b) for _, b in PLANTED_LINKS]
    occ = assign_follicle_cells(germ_bits[:, gcols], pool_bits[:, scols],
                                strength, N_SWAP_STEPS, rng)
    foll_bits = pool_bits[occ]
    return {"paired_bits": np.hstack([germ_bits, foll_bits]),        # columns follow ALL_GENES / qubit order
            "foll_ids": real["soma_ids"][world["soma_sel"]][occ]}


def report_cross_coupling(paired_bits, scenario):
    """Prints what the pairing actually contains: this is what training can learn from."""
    print(f"\n  Cross-population correlation (phi) from the {len(paired_bits)} contact pairs [{scenario}]")
    print("  rows = germline gene, columns = follicle gene")
    phi = np.corrcoef(paired_bits.T)[:3, 3:]
    hdr = "        " + "".join(f"{SHORT_NAME[g]:>8s}" for g in SOMATIC_GENES)
    print("  " + hdr)
    for i, g in enumerate(GERMLINE_GENES):
        print(f"  {SHORT_NAME[g]:>8s}" + "".join(f"{phi[i, j]:8.3f}" for j in range(3)))
    g_short = [SHORT_NAME[g] for g in GERMLINE_GENES]
    s_short = [SHORT_NAME[g] for g in SOMATIC_GENES]
    for a, b in PLANTED_LINKS:
        gi, si = g_short.index(a), s_short.index(b)
        gb, fb = paired_bits[:, gi], paired_bits[:, 3 + si]
        p1 = fb[gb == 1].mean()
        p0 = fb[gb == 0].mean()
        odds = ((p1 + 1e-9) / (1 - p1 + 1e-9)) / ((p0 + 1e-9) / (1 - p0 + 1e-9))
        print(f"  planted link {a}-{b}: P({b} ON | touching {a}-ON cell) = {p1:.3f}   "
              f"P({b} ON | touching {a}-OFF cell) = {p0:.3f}   odds ratio = {odds:.2f}")
    return phi


def save_pairs_csv(real, world, paired, path):
    rows = {
        "Chamber": world["chamber"] + 1,
        "Germline_Cell": real["germ_ids"][world["germ_sel"]],
        "Follicle_Cell": paired["foll_ids"],
        "Germline_x": world["germ_xy"][:, 0].round(3), "Germline_y": world["germ_xy"][:, 1].round(3),
        "Follicle_x": world["foll_xy"][:, 0].round(3), "Follicle_y": world["foll_xy"][:, 1].round(3),
        "Contact_distance": world["dist"].round(3),
    }
    for i, g in enumerate(ALL_GENES):
        rows[f"{SHORT_NAME[g]}_ON"] = paired["paired_bits"][:, i]
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f"[output] synthetic spatial pairs -> {path}")


def save_chamber_example(world, paired_by_scenario, path):
    """One chamber (the one with the most grk-ON germline cells), one panel per scenario."""
    a, b = PLANTED_LINKS[0]
    g_short = [SHORT_NAME[g] for g in GERMLINE_GENES]
    s_short = [SHORT_NAME[g] for g in SOMATIC_GENES]
    gi, si = g_short.index(a), s_short.index(b)
    any_bits = next(iter(paired_by_scenario.values()))["paired_bits"]
    per_chamber = np.bincount(world["chamber"], weights=any_bits[:, gi], minlength=world["n_chambers"])
    c = int(np.argmax(per_chamber))
    sl = np.where(world["chamber"] == c)[0]

    fig, axes = plt.subplots(1, len(paired_by_scenario), figsize=(6 * len(paired_by_scenario), 6))
    axes = np.atleast_1d(axes)
    for ax, (name, paired) in zip(axes, paired_by_scenario.items()):
        gb = paired["paired_bits"][sl, gi]
        fb = paired["paired_bits"][sl, 3 + si]
        for k, idx in enumerate(sl):
            matched = gb[k] == 1 and fb[k] == 1
            ax.plot([world["germ_xy"][idx, 0], world["foll_xy"][idx, 0]],
                    [world["germ_xy"][idx, 1], world["foll_xy"][idx, 1]],
                    color="#2e8b57" if matched else "#cccccc", lw=2.5 if matched else 1, zorder=1)
        ax.scatter(world["germ_xy"][sl, 0], world["germ_xy"][sl, 1], s=220, zorder=2,
                   c=["#e7a352" if v else "white" for v in gb], edgecolors="#8a5a20", linewidths=1.5)
        ax.scatter(world["foll_xy"][sl, 0], world["foll_xy"][sl, 1], s=220, marker="s", zorder=2,
                   c=["#5fa8d0" if v else "white" for v in fb], edgecolors="#2a5a80", linewidths=1.5)
        n_match = int(((gb == 1) & (fb == 1)).sum())
        ax.set_title(f"SYNTHETIC chamber {c + 1} - {name}\n"
                     f"{n_match} of {int(gb.sum())} {a}-ON germline cells (orange) touch a {b}-ON follicle cell (blue)",
                     fontsize=9)
        ax.set_aspect("equal")
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"[output] chamber example -> {path}")


# ============================================================================
# SECTION 2 -- ACTIVATION RATIOS -> Ry ENCODER ANGLES
# ============================================================================
def activation_ratios(binary_matrix):
    """p_j = fraction of cells in which gene j is active."""
    return binary_matrix.mean(axis=0)


def ry_angles_from_ratios(p):
    """theta_j = 2 * arcsin(sqrt(p_j))  -- so that P(qubit_j = 1) = p_j."""
    return 2.0 * np.arcsin(np.sqrt(np.clip(p, 0.0, 1.0)))


# ============================================================================
# SECTION 3 -- OBSERVED JOINT DISTRIBUTION P_obs OVER 64 STATES   (CHANGED)
# ============================================================================
def build_joint_observed_distribution(paired_bits):
    """
    P_obs counted DIRECTLY from the contact pairs: each pair is one observation
    of all 6 genes at once (3 germline genes from the germline cell, 3 follicle
    genes from the follicle cell touching it).

    The old version multiplied two independent distributions, which forced zero
    cross-population correlation. Counting real co-observed states keeps any
    cross-population structure that is present.
    Bit i of the state index is gene i of ALL_GENES (qubit 0 = least significant).
    """
    n_obs = paired_bits.shape[0]
    weights = 1 << np.arange(NGENES)                 # [1, 2, 4, 8, 16, 32]
    state_ids = paired_bits @ weights
    p_obs = np.bincount(state_ids, minlength=NSTATES) / n_obs

    if DROP_ZERO:
        p_obs[0] = 0.0
        p_obs = p_obs / p_obs.sum()

    return p_obs


# ============================================================================
# SECTION 4 -- QUANTUM GATE MATH (Ry, CRY, and their derivatives)
# ============================================================================
def ry_gate(theta):
    c, s = np.cos(theta / 2.0), np.sin(theta / 2.0)
    return np.array([[c, -s], [s, c]])


def der_ry_gate(theta):
    c, s = np.cos(theta / 2.0), np.sin(theta / 2.0)
    return 0.5 * np.array([[-s, -c], [c, -s]])


def _controlled_gate_matrix(nqubits, control, target, target_block):
    """Builds a full 2^n x 2^n matrix applying `target_block` to `target`
    only when `control` is |1>, identity elsewhere."""
    zero_proj = np.array([[1.0, 0.0], [0.0, 0.0]])
    one_proj  = np.array([[0.0, 0.0], [0.0, 1.0]])
    RR, II = None, None
    for i in range(nqubits):
        if i == control:
            fac_RR, fac_II = one_proj, zero_proj
        elif i == target:
            fac_RR, fac_II = target_block, np.identity(2)
        else:
            fac_RR, fac_II = np.identity(2), np.identity(2)
        RR = np.kron(fac_RR, RR) if RR is not None else fac_RR
        II = np.kron(fac_II, II) if II is not None else fac_II
    return RR, II


def cry_gate(theta, nqubits, control, target):
    RR, II = _controlled_gate_matrix(nqubits, control, target, ry_gate(theta))
    return RR + II


def der_cry_gate(theta, nqubits, control, target):
    RR, _ = _controlled_gate_matrix(nqubits, control, target, der_ry_gate(theta))
    return RR


def tensor_product(matrices):
    """matrices given in qubit-index order [q0, q1, ..., q_{n-1}]."""
    arr = None
    for m in matrices:
        arr = np.kron(m, arr) if arr is not None else m
    return arr


def matrix_multiplication(matrices):
    """Given [T1, T2, ..., Tk] (applied in that order to the state),
    returns Tk @ ... @ T2 @ T1."""
    arr = None
    for m in matrices:
        arr = m @ arr if arr is not None else m
    return arr


# ============================================================================
# SECTION 5 -- CANDIDATE EDGES: FULLY-CONNECTED, 30 DIRECTED CRY GATES
# ============================================================================
def build_edges(genes):
    """All ordered pairs (g_i, g_j), i != j  ->  n^2 - n edges (30 for n=6)."""
    return [(a, b) for a in genes for b in genes if a != b]


# ============================================================================
# SECTION 6 -- TRAINING: exact analytic gradient descent
# ============================================================================
def laplace_smooth(distribution, ncells, alpha=LAPLACE_ALPHA):
    dim = len(distribution)
    scaled = ncells * distribution
    n_total = scaled.sum()
    scaled = scaled + alpha
    n_total = n_total + alpha * dim
    return scaled / n_total, n_total


def loss_constraint(phi_values, ratio):
    """Boundary-avoidance regularizer, keeping edge angles away from the
    +/- pi/2 singularity."""
    return np.sum(ratio / np.power(phi_values ** 4 - (np.pi / 2) ** 4, 2))


def train(p_obs, theta_fixed, edges, indexes, ncells_for_smoothing,
          epochs=EPOCHS, lr=LEARNING_RATE):
    """
    theta_fixed : length-6 array, the Ry encoder angles (NOT trained).
    edges       : list of 30 (gene_a, gene_b) tuples.
    indexes     : list of 30 (control_qubit, target_qubit) tuples.
    Returns the trained phi array (length 30) and the loss history.
    """
    n_edges = len(edges)
    phi = np.zeros(n_edges)   # start with NO assumed interactions

    encoder_mats = [ry_gate(theta_fixed[i]) for i in range(NGENES)]
    encoder = tensor_product(encoder_mats)          # fixed throughout training
    input_state = np.zeros((NSTATES, 1))
    input_state[0, 0] = 1.0

    p_obs_col = p_obs.reshape(NSTATES, 1)
    loss_history = []

    for epoch in range(epochs):
        # ---- forward pass: exact statevector ----
        regulation_mats = [cry_gate(phi[i], NGENES, indexes[i][0], indexes[i][1])
                            for i in range(n_edges)]
        transform_reg = matrix_multiplication(regulation_mats)
        v = transform_reg @ encoder @ input_state             # (64,1) amplitude vector
        p_out = v ** 2

        if DROP_ZERO:
            p_out = p_out.copy()
            p_out[0, 0] = 0.0
            p_out = p_out / p_out.sum()

        # ---- Laplace-smoothed probabilities for a numerically stable KL ----
        h_p_out, h_N_out = laplace_smooth(p_out.flatten(), ncells_for_smoothing)
        h_p_obs, h_N_obs = laplace_smooth(p_obs_col.flatten(), ncells_for_smoothing)
        h_p_out = h_p_out.reshape(NSTATES, 1)
        h_p_obs = h_p_obs.reshape(NSTATES, 1)

        loss = float(np.sum(h_p_out * np.log(h_p_out / h_p_obs)))     # D_KL(model || obs)
        loss_ratio = loss / loss_constraint(phi, ratio=1.0)
        cons = loss_constraint(phi, ratio=loss_ratio)
        loss_history.append(loss)

        # ---- analytic gradient ----
        log_term = np.log(h_p_out / h_p_obs)
        der_const = (-2 * loss_ratio * 4 * phi ** 3) / np.power(phi ** 4 - (np.pi / 2) ** 4, 3)
        gradient = np.zeros(n_edges)

        for i in range(n_edges):
            control, target = indexes[i]
            der_reg_mats = regulation_mats.copy()
            der_reg_mats[i] = der_cry_gate(phi[i], NGENES, control, target)
            der_transform_reg = matrix_multiplication(der_reg_mats)
            der_state = der_transform_reg @ encoder @ input_state       # (64,1)

            der_loss = (1 + log_term) * 2 * v * der_state * (NSTATES / h_N_out)
            gradient[i] = np.sum(der_loss) + der_const[i]

        # ---- early stopping on relative loss change ----
        if epoch >= LOSS_WINDOW + 1:
            recent = np.mean(loss_history[epoch - (LOSS_WINDOW - 1): epoch + 1])
            past = loss_history[epoch - LOSS_WINDOW]
            delta = abs(past - recent) / abs(recent) if recent != 0 else 0.0
            if delta < LOSS_REL_THRESHOLD:
                print(f"[train] Converged at epoch {epoch} (delta={delta:.2e})")
                break

        # ---- gradient descent step ----
        phi = phi - lr * gradient

        if epoch % 100 == 0:
            print(f"[train] epoch {epoch:4d}  KL loss = {loss:.6f}")

    print(f"[train] final KL loss = {loss_history[-1]:.6f} after {len(loss_history)} epochs")
    return phi, loss_history


# ============================================================================
# SECTION 7 -- REBUILD IN REAL QISKIT & RUN ON A FREE SIMULATOR (AerSimulator)
# ============================================================================
def build_qiskit_circuit(theta_fixed, phi_trained, edges, indexes):
    """Builds the actual, literal Qiskit circuit: 6 Ry gates (encoder) +
    30 trained CRY gates (regulation) + measurement, all 6 qubits."""
    circ = QuantumCircuit(NGENES, NGENES)

    for q in range(NGENES):
        circ.ry(theta_fixed[q], q)
    circ.barrier()

    for i, (control, target) in enumerate(indexes):
        circ.cry(phi_trained[i], control, target)
    circ.barrier()

    circ.measure(range(NGENES), range(NGENES))
    return circ


def run_on_simulator(circ, shots=SHOTS):
    """Executes the circuit on Qiskit's free, local AerSimulator."""
    backend = AerSimulator()
    compiled = transpile(circ, backend)
    job = backend.run(compiled, shots=shots)
    counts = job.result().get_counts()
    return counts


def counts_to_probabilities(counts, shots):
    """Qiskit bitstrings are ordered qN-1...q1q0 (qubit 0 = rightmost
    character), matching our state-index convention directly."""
    p_model = np.zeros(NSTATES)
    for bitstring, n in counts.items():
        idx = int(bitstring, 2)
        p_model[idx] = n / shots
    return p_model


# ============================================================================
# SECTION 8a -- OUTPUT: FULL CIRCUIT DIAGRAM
# ============================================================================
def save_circuit_diagram(circ, path):
    fig = circ.draw(output="mpl", fold=-1, style={"fontsize": 10})
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"[output] full circuit diagram -> {path}")


# ============================================================================
# SECTION 8b -- OUTPUT: WEIGHTED GRN NETWORK DIAGRAM
# ============================================================================
def save_weighted_network(edges, phi_trained, path, dataset_name):
    G = nx.DiGraph()
    for gene in ALL_GENES:
        G.add_node(SHORT_NAME[gene])

    max_abs_phi = np.max(np.abs(phi_trained)) if np.max(np.abs(phi_trained)) > 0 else 1.0

    pos = {}
    germ_short = [SHORT_NAME[g] for g in GERMLINE_GENES]
    som_short  = [SHORT_NAME[g] for g in SOMATIC_GENES]
    for i, g in enumerate(germ_short):
        pos[g] = (-1.0, 1.0 - i)
    for i, g in enumerate(som_short):
        pos[g] = (1.0, 1.0 - i)

    fig, ax = plt.subplots(figsize=(9, 7))

    node_colors = ["#e7a352" if g in germ_short else "#5fa8d0" for g in G.nodes()]
    nx.draw_networkx_nodes(G, pos, node_size=1800, node_color=node_colors,
                            edgecolors="#20242c", linewidths=1.5, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=12, font_weight="bold", ax=ax)

    for (gene_a, gene_b), phi_val in zip(edges, phi_trained):
        a, b = SHORT_NAME[gene_a], SHORT_NAME[gene_b]
        magnitude = abs(phi_val)
        alpha = float(np.clip((magnitude / max_abs_phi) ** 1.3, 0.02, 1.0))
        width = float(np.clip(0.4 + 5.0 * (magnitude / max_abs_phi), 0.4, 5.4))
        same_population = (gene_a in GERMLINE_GENES) == (gene_b in GERMLINE_GENES)
        rad = 0.15 if same_population else 0.08
        color = "#c96a3a" if phi_val >= 0 else "#3a7fc9"

        nx.draw_networkx_edges(
            G, pos, edgelist=[(a, b)], ax=ax,
            width=width, alpha=alpha, edge_color=color,
            connectionstyle=f"arc3,rad={rad}",
            arrowstyle="-|>", arrowsize=14, node_size=1800,
        )

    ax.set_title(f"Inferred gene regulatory network — {dataset_name}\n"
                 f"(edge weight = |trained CRY angle|)", fontsize=12)
    ax.text(-1.0, 1.8, "germline", ha="center", fontsize=10, color="#8a5a20")
    ax.text(1.0, 1.8, "somatic", ha="center", fontsize=10, color="#2a5a80")
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"[output] weighted network diagram -> {path}")


# ============================================================================
# SECTION 8c -- OUTPUT: TRAINING CURVE + EDGE TABLE
# ============================================================================
def save_training_curve(loss_history, path, dataset_name):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(loss_history, color="#5fa8d0")
    ax.set_xlabel("epoch")
    ax.set_ylabel("D_KL(P_model || P_obs)")
    ax.set_title(f"Training loss — {dataset_name}")
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"[output] training curve -> {path}")


def save_edge_table(edges, phi_trained, path, dataset_name):
    threshold = np.percentile(np.abs(phi_trained), EDGE_PERCENTILE_FOR_BINARY)
    rows = []
    for (gene_a, gene_b), phi_val in zip(edges, phi_trained):
        a_short, b_short = SHORT_NAME[gene_a], SHORT_NAME[gene_b]
        above_threshold = abs(phi_val) >= threshold
        is_expected = (a_short, b_short) in EXPECTED_EDGES

        # Validation logic:
        #   expected edge  + model says "connected"    -> Correct (found a real connection)
        #   expected edge  + model says "not connected" -> Incorrect (missed a real connection)
        #   unexpected edge + model says "not connected" -> Correct (correctly found no relation)
        #   unexpected edge + model says "connected"    -> Incorrect (flagged a connection that isn't known to exist)
        if is_expected:
            validation = "Correct" if above_threshold else "Incorrect"
        else:
            validation = "Correct" if not above_threshold else "Incorrect"

        rows.append({
            "control_gene": a_short,
            "target_gene": b_short,
            "edge": f"{a_short} -> {b_short}",
            "population_pair": (
                "intra-germline" if gene_a in GERMLINE_GENES and gene_b in GERMLINE_GENES else
                "intra-somatic" if gene_a in SOMATIC_GENES and gene_b in SOMATIC_GENES else
                "cross-population"
            ),
            "trained_phi_radians": phi_val,
            "abs_phi": abs(phi_val),
            f"above_{EDGE_PERCENTILE_FOR_BINARY}th_percentile": above_threshold,
            "known_biological_relationship": EXPECTED_EDGES.get((a_short, b_short), "(none known)"),
            "validation": validation,
        })
    out = pd.DataFrame(rows).sort_values("abs_phi", ascending=False)

    score = int((out["validation"] == "Correct").sum())
    n_total = len(out)

    out.to_csv(path, index=False)
    print(f"[output] all 30 learned edge angles -> {path}")
    print(out.to_string(index=False))
    print(f"\n[validation] {dataset_name}: {score}/{n_total} edges match known biology "
          f"(score = {score}/{n_total})")
    return out, score


# ============================================================================
# RUN ONE SCENARIO, START TO FINISH   (same steps as before; changes marked NEW)
# ============================================================================
def run_pipeline_for_scenario(scenario, paired, n_pairs):
    dataset_name = f"FlyCellAtlas_SYNTHETIC_{scenario}"
    paired_bits = paired["paired_bits"]
    print("\n" + "#" * 70)
    print(f"# SCENARIO: {scenario}   (SYNTHETIC spatial data built from real cells)")
    print(f"# INPUT:    {INPUT_XLSX}")
    print("#" * 70)

    out_circuit  = os.path.join(BASE_FOLDER, f"quantum_circuit_diagram_{dataset_name}.png")
    out_network  = os.path.join(BASE_FOLDER, f"weighted_grn_network_{dataset_name}.png")
    out_curve    = os.path.join(BASE_FOLDER, f"training_loss_curve_{dataset_name}.png")
    out_edges    = os.path.join(BASE_FOLDER, f"learned_edge_angles_{dataset_name}.csv")

    print("\n" + "=" * 70)
    print("STEP 1: synthetic spatial data (real cells, synthetic contacts)")
    print("=" * 70)
    print(f"{n_pairs} contact pairs")
    report_cross_coupling(paired_bits, scenario)                               # NEW

    print("\n" + "=" * 70)
    print("STEP 2: activation ratios -> Ry encoder angles (fixed, from data)")
    print("=" * 70)
    p_all = activation_ratios(paired_bits)            # order matches ALL_GENES   (CHANGED: from the pairs)
    theta_fixed = ry_angles_from_ratios(p_all)
    for gene, p, th in zip(ALL_GENES, p_all, theta_fixed):
        print(f"  {SHORT_NAME[gene]:5s}  p={p:.4f}   theta={th:.4f} rad")

    print("\n" + "=" * 70)
    print("STEP 3: building the observed joint distribution P_obs (64 states)")
    print("=" * 70)
    p_obs = build_joint_observed_distribution(paired_bits)                      # CHANGED
    print(f"P_obs sums to {p_obs.sum():.6f}, nonzero states: {np.count_nonzero(p_obs)}/64")

    print("\n" + "=" * 70)
    print("STEP 4: building the 30 fully-connected candidate CRY edges")
    print("=" * 70)
    edges = build_edges(ALL_GENES)
    indexes = [(ALL_GENES.index(a), ALL_GENES.index(b)) for a, b in edges]
    print(f"{len(edges)} candidate directed edges (expected 30)")

    print("\n" + "=" * 70)
    print("STEP 5: training (exact analytic gradient descent, qscGRN-style)")
    print("=" * 70)
    ncells_for_smoothing = n_pairs                                              # CHANGED
    phi_trained, loss_history = train(p_obs, theta_fixed, edges, indexes,
                                       ncells_for_smoothing)

    print("\n" + "=" * 70)
    print("STEP 6: rebuilding the trained circuit in Qiskit and running it")
    print("         on the free, local AerSimulator")
    print("=" * 70)
    circ = build_qiskit_circuit(theta_fixed, phi_trained, edges, indexes)
    counts = run_on_simulator(circ, shots=SHOTS)
    p_model_sampled = counts_to_probabilities(counts, SHOTS)
    print(f"Executed circuit with {SHOTS} shots on AerSimulator; "
          f"{len(counts)} distinct states observed.")

    print("\n" + "=" * 70)
    print("STEP 7: saving outputs")
    print("=" * 70)
    save_circuit_diagram(circ, out_circuit)
    save_weighted_network(edges, phi_trained, out_network, dataset_name)
    save_training_curve(loss_history, out_curve, dataset_name)
    edge_table, validation_score = save_edge_table(edges, phi_trained, out_edges, dataset_name)

    print(f"\nDone with {dataset_name}. Outputs written:")
    print(f"  - {out_circuit}")
    print(f"  - {out_network}")
    print(f"  - {out_curve}")
    print(f"  - {out_edges}")

    return {
        "pairs": n_pairs,
        "final_kl_loss": loss_history[-1],
        "phi_trained": phi_trained,
        "edge_table": edge_table,
        "validation_score": validation_score,
    }


# ============================================================================
# COMPARISON (NEW) -- what changed between the scenarios
# ============================================================================
def summarize_edges(edge_table):
    et = edge_table
    planted = set()
    for a, b in PLANTED_LINKS:
        planted |= {(a, b), (b, a)}
    is_planted = [(c, t) in planted for c, t in zip(et["control_gene"], et["target_gene"])]
    cross_other = et[(et["population_pair"] == "cross-population") & ~np.array(is_planted)]
    out = {}
    pct_col = f"above_{EDGE_PERCENTILE_FOR_BINARY}th_percentile"
    for a, b in PLANTED_LINKS:
        for c, t in ((a, b), (b, a)):
            row = et[(et["control_gene"] == c) & (et["target_gene"] == t)].iloc[0]
            out[f"phi {c}->{t}"] = round(float(row["trained_phi_radians"]), 3)
            out[f"{c}->{t} above {EDGE_PERCENTILE_FOR_BINARY}th pct"] = bool(row[pct_col])
    out["strongest OTHER cross-population |phi|"] = round(float(cross_other["abs_phi"].max()), 3)
    known = et[et["known_biological_relationship"] != "(none known)"]
    out["known edges found (of 8)"] = int((known["validation"] == "Correct").sum())
    out["validation score (of 30)"] = int((et["validation"] == "Correct").sum())
    return out


def intra_population_counts(paired_bits):
    w = 1 << np.arange(3)
    return (np.bincount(paired_bits[:, :3] @ w, minlength=8),
            np.bincount(paired_bits[:, 3:] @ w, minlength=8))


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 70)
    print("LOADING REAL CELLS")
    print("=" * 70)
    real = load_panel(INPUT_XLSX)
    print(f"germline cells: {len(real['germ_ids'])}   somatic cells: {len(real['soma_ids'])}")

    print("\n" + "=" * 70)
    print("BUILDING SYNTHETIC EGG CHAMBERS (geometry only; expression is never touched)")
    print("=" * 70)
    world = build_synthetic_world(real, np.random.default_rng(SEED))
    print(f"{world['n_chambers']} chambers x {PAIRS_PER_CHAMBER} contact pairs = {world['n_pairs']} pairs "
          f"(mean contact distance {world['dist'].mean():.2f})")
    print(f"planted link(s): {PLANTED_LINKS}   coupling strength (planted scenario): {COUPLING_STRENGTH}")

    paired = {}
    for i, (name, strength) in enumerate(SCENARIOS.items()):
        print(f"\nBuilding contacts for scenario '{name}' (strength {strength}) ...")
        paired[name] = make_paired_observations(real, world, strength,
                                                np.random.default_rng(SEED + 1 + i))
        save_pairs_csv(real, world, paired[name],
                       os.path.join(BASE_FOLDER, f"synthetic_spatial_pairs_{name}.csv"))
    save_chamber_example(world, paired, os.path.join(BASE_FOLDER, "synthetic_chamber_example.png"))

    # Sanity check: the scenarios must differ ONLY in who touches whom
    names = list(paired)
    base_g, base_s = intra_population_counts(paired[names[0]]["paired_bits"])
    same_intra = all(np.array_equal(base_g, intra_population_counts(paired[n]["paired_bits"])[0]) and
                     np.array_equal(base_s, intra_population_counts(paired[n]["paired_bits"])[1])
                     for n in names[1:])
    same_marg = all(np.array_equal(paired[names[0]]["paired_bits"].sum(0), paired[n]["paired_bits"].sum(0))
                    for n in names[1:])
    print(f"\nCheck - identical gene activation counts across scenarios: {same_marg}")
    print(f"Check - identical within-germline and within-follicle distributions across scenarios: {same_intra}")

    results = {name: run_pipeline_for_scenario(name, paired[name], world["n_pairs"]) for name in SCENARIOS}

    print("\n\n" + "#" * 70)
    print("# SYNTHETIC SPATIAL TEST COMPLETE")
    print("#" * 70)
    print(pd.DataFrame([{
        "Scenario": n, "Contact pairs": r["pairs"], "Final KL loss": round(r["final_kl_loss"], 6),
        "Validation score (out of 30)": r["validation_score"],
    } for n, r in results.items()]).to_string(index=False))

    print("\nWhat the trained circuit did with the planted link and with every other cross-population edge:")
    comp = pd.DataFrame({n: summarize_edges(r["edge_table"]) for n, r in results.items()})
    print(comp.to_string())

if __name__ == "__main__":
    main()

LOADING REAL CELLS
germline cells: 3267   somatic cells: 3425

BUILDING SYNTHETIC EGG CHAMBERS (geometry only; expression is never touched)
204 chambers x 16 contact pairs = 3264 pairs (mean contact distance 1.01)
planted link(s): [('grk', 'top')]   coupling strength (planted scenario): 2.0

Building contacts for scenario 'null' (strength 0.0) ...
[output] synthetic spatial pairs -> C:\Users\13306\Desktop\CS-495\synthetic_spatial_pairs_null.csv

Building contacts for scenario 'planted' (strength 2.0) ...
[output] synthetic spatial pairs -> C:\Users\13306\Desktop\CS-495\synthetic_spatial_pairs_planted.csv
[output] chamber example -> C:\Users\13306\Desktop\CS-495\synthetic_chamber_example.png

Check - identical gene activation counts across scenarios: True
Check - identical within-germline and within-follicle distributions across scenarios: True

######################################################################
# SCENARIO: null   (SYNTHETIC spatial data built from real cells)
# INPU